# 05 — Forecasting Inference Demo
This notebook is the final, standalone-study step for the Nottingham monthly temperature
forecasting study. It is a **consumer-only, read-only** demonstration of the frozen final
forecasting model produced and authenticated by Notebook 04. It performs no training, no
model selection, no refit, and no final-holdout evaluation.

## 1. Inference Context and Consumer-Only Boundary
Notebook 05 consumes exactly one authenticated entry boundary:
`artifacts/models/nottem/final-model-handoff.json`. Everything downstream (the inference
bundle, the frozen model, the forecast) is reached only through the hardened, read-only
loaders in `scripts.forecasting_finalization` and the small consumer helper in
`scripts.forecasting_inference`.

This notebook does not:
- fit, refit, retune, or retrain any model,
- reopen model selection or backtesting,
- open or read the final holdout (`data/processed/nottem/final-holdout.csv`),
- write, mutate, or create any artifact under `artifacts/models/nottem/`.

It is executed here in a fresh kernel/process: the first operational cell below reconstructs
the project context, the authenticated handoff, the inference bundle, and the frozen model
from scratch -- nothing is reused from Notebook 04's own kernel state.

In [18]:
from pathlib import Path
import hashlib
import subprocess
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'scripts/project_context.py').is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from scripts.project_context import get_project_context

PROJECT = get_project_context(start=PROJECT_ROOT)
HANDOFF_PATH = "artifacts/models/nottem/final-model-handoff.json"

print({
    "project_root": str(PROJECT.root),
    "handoff_path": HANDOFF_PATH,
    "boundary": "read-only inference consumer; no training, no model selection, no final holdout access, no artifact mutation",
})

{'project_root': '/home/fabyuu/Projetos/DATASET-ANALISYS/dataset-study-nottingham-monthly-temperatures', 'handoff_path': 'artifacts/models/nottem/final-model-handoff.json', 'boundary': 'read-only inference consumer; no training, no model selection, no final holdout access, no artifact mutation'}


## 2. Authenticated Final-Model Handoff
The official entry loader, `load_and_validate_forecasting_final_model_handoff`, authenticates
the Notebook 03 lineage, the selected candidate/role/family/specification, the frozen-model
byte SHA and semantic state fingerprint, the final-model manifest, the final-test evidence,
the inference bundle, and the final metrics -- and replays the frozen model's forecast against
the persisted final evidence. If this loader raises, this notebook stops: there is no fallback
and no direct `joblib.load`.

Immediately after authentication, this notebook snapshots the byte SHA-256 of all five final
artifacts, to be re-checked at the end of the notebook (Section 14).

In [19]:
from scripts.forecasting_finalization import load_and_validate_forecasting_final_model_handoff
from scripts.forecasting_inference import (
    ForecastingInferenceError,
    load_authenticated_forecasting_consumer,
    normalize_forecasting_inference_history,
    forecast_from_history,
    validate_forecasting_inference_output,
)

handoff = load_and_validate_forecasting_final_model_handoff(project_root=PROJECT.root, handoff_path=HANDOFF_PATH)

print({
    "schema_version": handoff["schema_version"],
    "selected_candidate_id": handoff["selected_model"]["candidate_id"],
    "selected_role": handoff["selected_model"]["role"],
    "selected_family": handoff["selected_model"]["family"],
    "inference_demo_ready": handoff["readiness"]["inference_demo_ready"],
    "operational_modeling_ready": handoff["readiness"]["operational_modeling_ready"],
})

{'schema_version': 'forecasting-final-model-handoff.v1', 'selected_candidate_id': 'seasonal_trend_ols', 'selected_role': 'candidate', 'selected_family': 'DeterministicSeasonalTrendOLS', 'inference_demo_ready': True, 'operational_modeling_ready': False}


In [20]:
FINAL_ARTIFACT_DIR = PROJECT.root / "artifacts/models/nottem"
FINAL_ARTIFACT_NAMES = [
    "final-pipeline.joblib",
    "final-model-manifest.json",
    "final-test-evidence.json",
    "inference-bundle.json",
    "final-model-handoff.json",
]


def artifact_hash_snapshot() -> dict:
    return {
        name: hashlib.sha256((FINAL_ARTIFACT_DIR / name).read_bytes()).hexdigest()
        for name in FINAL_ARTIFACT_NAMES
    }


artifact_hashes_before = artifact_hash_snapshot()
print(artifact_hashes_before)

{'final-pipeline.joblib': '3060d7a351688d7e60622d9112dbe01350e97964a01934b102beff912ae36f41', 'final-model-manifest.json': 'c6859e1425bbbf675ba169243e855ef153b6709c454341cdc75c1de14015fefa', 'final-test-evidence.json': 'a46f34bab5c652f186f5b4055d7b1d018823a631ae8f9f2cf1e042f248c1ba22', 'inference-bundle.json': 'a74d8d67256550694f12131a295a018511ae0ef476c12b6bb2dc72cb013b6079', 'final-model-handoff.json': '3a280491f5fa9e7d51db8e01c09dd202a2beddd3980f5074592fb44ceaa70732'}


## 3. Authenticated Forecasting Inference Bundle
`load_authenticated_forecasting_consumer` composes the hardened handoff loader, the hardened
bundle loader (`load_and_validate_forecasting_inference_bundle`), and the trusted frozen-model
loader (`load_trusted_forecasting_model_from_bundle`) into one read-only consumer, and
cross-checks candidate/family/specification/state-fingerprint coherence between all three. The
bundle path is derived from the authenticated handoff's own sibling reference -- never a second,
independently hardcoded path.

In [21]:
consumer = load_authenticated_forecasting_consumer(project_root=PROJECT.root, handoff_path=HANDOFF_PATH)
bundle = consumer.inference_bundle

print({
    "target": bundle["target"],
    "time": bundle["time"],
    "input_contract_required_columns": bundle["input_contract"]["required_columns"],
    "input_contract_period_requirements": bundle["input_contract"]["period_requirements"],
    "input_contract_exogenous_predictors": bundle["input_contract"]["exogenous_predictors"],
    "output_contract": bundle["output_contract"],
})

{'target': {'column': 'temperature', 'dtype': 'numeric finite', 'semantics': 'Monthly average air temperature at Nottingham Castle', 'unit': 'degrees Fahrenheit'}, 'time': {'expected_frequency': 'M', 'forecast_horizon': 12, 'forecast_origin': 'last period of supplied history', 'seasonal_period': 12}, 'input_contract_required_columns': ['period', 'temperature'], 'input_contract_period_requirements': {'contiguous': True, 'monotonic_increasing': True, 'monthly': True, 'unique': True}, 'input_contract_exogenous_predictors': 'none', 'output_contract': {'columns': ['period', 'forecast'], 'forecast_dtype': 'numeric finite', 'periods': 'origin+1 through origin+12', 'row_count': 12, 'unit': 'degrees Fahrenheit'}}


## 4. Trusted Frozen-Model Loading
The frozen model was loaded only through `load_trusted_forecasting_model_from_bundle`, which
verifies the model artifact's byte SHA-256 *before* any deserialization, then validates the
deserialized object's schema and semantic state fingerprint. No `joblib.load` call is made
directly by this notebook or by `scripts.forecasting_inference`.

In [22]:
model = consumer.model

print({
    "schema_version": model.schema_version,
    "candidate_id": model.candidate_id,
    "family": model.family,
    "training_end": model.training_end,
    "model_state_semantic_fingerprint": model.model_state_semantic_fingerprint,
    "coefficient_count": model.coefficient_count,
    "fitted": model.fitted,
    "frozen": model.frozen,
})

{'schema_version': 'forecasting-frozen-model.v1', 'candidate_id': 'seasonal_trend_ols', 'family': 'DeterministicSeasonalTrendOLS', 'training_end': '1938-12', 'model_state_semantic_fingerprint': 'ddb57634c40d4f11192ca0478d40904a8d00e0e20f3aee679b46dc1117501279', 'coefficient_count': 13, 'fitted': True, 'frozen': True}


## 5. Forecasting Input Contract
The executable input contract accepts exactly one representation: a `pandas.DataFrame` with
columns `["period", "temperature"]`, in that exact order, at least one row, monthly/monotonic/
unique/contiguous periods, and finite numeric-coercible temperatures ending at or after the
frozen training end. There are no exogenous predictors and no refit-on-input: supplied
temperature values establish the forecast origin and validate chronology, they never update the
frozen coefficients.

In [23]:
print({
    "required_columns_in_order": ["period", "temperature"],
    "minimum_history_observations": bundle["input_contract"]["minimum_history_observations"],
    "history_role": bundle["input_contract"]["history_role"],
    "exogenous_predictors": bundle["input_contract"]["exogenous_predictors"],
    "refit_on_input": bundle["input_contract"]["refit_on_input"],
    "post_training_history_limitation": bundle["input_contract"]["post_training_history_limitation"],
})

{'required_columns_in_order': ['period', 'temperature'], 'minimum_history_observations': 1, 'history_role': 'establish forecast origin and validate monthly chronology', 'exogenous_predictors': 'none', 'refit_on_input': False, 'post_training_history_limitation': 'values establish origin/context only; frozen coefficients are not updated'}


## 6. Historical-Series Normalization and Validation
The cell below normalizes one valid synthetic history, then demonstrates several contract
rejections. None of this reads training or holdout data from disk; every history here is
constructed in-memory for pedagogical purposes.

In [24]:
canonical_history = pd.DataFrame({"period": ["1938-12"], "temperature": [50.0]})
normalized = normalize_forecasting_inference_history(consumer, canonical_history)
print(normalized)

invalid_examples = {
    "missing temperature column": pd.DataFrame({"period": ["1938-12"]}),
    "duplicate period": pd.DataFrame({"period": ["1938-12", "1938-12"], "temperature": [50.0, 51.0]}),
    "monthly gap": pd.DataFrame({"period": ["1938-10", "1938-12"], "temperature": [50.0, 51.0]}),
    "non-numeric temperature": pd.DataFrame({"period": ["1938-12"], "temperature": ["warm"]}),
    "non-finite temperature": pd.DataFrame({"period": ["1938-12"], "temperature": [float("inf")]}),
    "history ends before the frozen training end": pd.DataFrame({"period": ["1938-11"], "temperature": [50.0]}),
}
rejections = {}
for label, frame in invalid_examples.items():
    try:
        normalize_forecasting_inference_history(consumer, frame)
        rejections[label] = "unexpectedly accepted"
    except Exception as exc:
        rejections[label] = type(exc).__name__
print(rejections)

period
1938-12    50.0
Freq: M, Name: temperature, dtype: float64
{'missing temperature column': 'ForecastingFinalizationContractError', 'duplicate period': 'ForecastingFinalizationContractError', 'monthly gap': 'ForecastingFinalizationContractError', 'non-numeric temperature': 'ForecastingFinalizationContractError', 'non-finite temperature': 'ForecastingFinalizationContractError', 'history ends before the frozen training end': 'ForecastingFinalizationContractError'}


## 7. Canonical 12-Month Forecast Demo
The history below is a **contract demonstration input, NOT observed source data**: a single
synthetic row at the frozen training end (`1938-12`) with a deterministic finite placeholder
temperature of `50.0`. It exists only to satisfy the history schema and establish the forecast
origin; it is not an observed Nottingham reading. The resulting forecast covers `1939-01` through
`1939-12`, with no `y_true`, no holdout access, and no scoring.

In [25]:
canonical_history = pd.DataFrame({"period": ["1938-12"], "temperature": [50.0]})
canonical_forecast = forecast_from_history(consumer, canonical_history)
print(canonical_forecast)

     period   forecast
0   1939-01  40.314766
1   1939-02  39.704240
2   1939-03  42.788450
3   1939-04  46.814766
4   1939-05  53.172661
5   1939-06  58.646345
6   1939-07  62.567398
7   1939-08  61.056871
8   1939-09  56.993713
9   1939-10  50.246345
10  1939-11  42.972661
11  1939-12  40.225292


## 8. Later-Origin Forecast Demo
This second history is also explicitly synthetic: three placeholder rows ending `1940-12`, used
only to demonstrate that the forecast origin can advance well past the frozen training end
without any refit. The resulting forecast covers `1941-01` through `1941-12`, produced by the
same frozen coefficients as Section 7.

In [26]:
later_history = pd.DataFrame({
    "period": ["1940-10", "1940-11", "1940-12"],
    "temperature": [50.0, 50.0, 50.0],
})
later_forecast = forecast_from_history(consumer, later_history)
print(later_forecast)

model_state_unchanged = model.model_state_semantic_fingerprint == consumer.model.model_state_semantic_fingerprint
print({"model_state_unchanged_after_later_origin_forecast": model_state_unchanged})

     period   forecast
0   1941-01  40.435614
1   1941-02  39.825088
2   1941-03  42.909298
3   1941-04  46.935614
4   1941-05  53.293509
5   1941-06  58.767193
6   1941-07  62.688246
7   1941-08  61.177719
8   1941-09  57.114561
9   1941-10  50.367193
10  1941-11  43.093509
11  1941-12  40.346140
{'model_state_unchanged_after_later_origin_forecast': True}


## 9. Forecast-Origin and History-Value Semantics
The forecast origin is always the last validated historical period; the future window is always
`origin + 1` through `origin + horizon`. For the current authenticated winner,
`seasonal_trend_ols` (`DeterministicSeasonalTrendOLS`), the frozen coefficients are not updated
by the supplied temperature values -- only the chronology (the origin) matters. The cell below
demonstrates this by holding the origin fixed and varying the supplied temperatures by a wide
margin.

This is a **model-specific** property of the currently selected frozen OLS specification, not a
universal forecasting rule.

In [27]:
origin_a_periods = list(canonical_forecast["period"])
origin_b_periods = list(later_forecast["period"])

low_history = pd.DataFrame({
    "period": ["1940-10", "1940-11", "1940-12"],
    "temperature": [20.0, 20.0, 20.0],
})
high_history = pd.DataFrame({
    "period": ["1940-10", "1940-11", "1940-12"],
    "temperature": [80.0, 80.0, 80.0],
})
low_forecast = forecast_from_history(consumer, low_history)
high_forecast = forecast_from_history(consumer, high_history)
history_value_invariant = low_forecast.equals(high_forecast)

print({
    "origin_a_periods": origin_a_periods,
    "origin_b_periods": origin_b_periods,
    "periods_differ_by_origin": origin_a_periods != origin_b_periods,
    "history_value_invariant_for_seasonal_trend_ols": history_value_invariant,
})

{'origin_a_periods': ['1939-01', '1939-02', '1939-03', '1939-04', '1939-05', '1939-06', '1939-07', '1939-08', '1939-09', '1939-10', '1939-11', '1939-12'], 'origin_b_periods': ['1941-01', '1941-02', '1941-03', '1941-04', '1941-05', '1941-06', '1941-07', '1941-08', '1941-09', '1941-10', '1941-11', '1941-12'], 'periods_differ_by_origin': True, 'history_value_invariant_for_seasonal_trend_ols': True}


## 10. Deterministic Repeatability
Calling the frozen model twice on the same valid history must produce exactly the same periods
and numerically identical predictions.

In [28]:
repeat_1 = forecast_from_history(consumer, canonical_history)
repeat_2 = forecast_from_history(consumer, canonical_history)

deterministic = list(repeat_1["period"]) == list(repeat_2["period"]) and np.allclose(
    repeat_1["forecast"].to_numpy(dtype=float),
    repeat_2["forecast"].to_numpy(dtype=float),
    rtol=1e-12,
    atol=1e-12,
)
print({"deterministic_repeatability": deterministic})

{'deterministic_repeatability': True}


## 11. No-Refit and Model-Immutability Audit
The frozen model's state descriptor, semantic state fingerprint, and coefficient vector are
captured before and after several forecast calls (including the different-origin call from
Section 8). Every one of them must be exactly unchanged: the consumer path never mutates the
frozen model.

In [29]:
descriptor_before = model.state_descriptor()
fingerprint_before = model.model_state_semantic_fingerprint
coefficients_before = tuple(model.coefficients)

forecast_from_history(consumer, canonical_history)
forecast_from_history(consumer, later_history)
forecast_from_history(consumer, low_history)

descriptor_unchanged = model.state_descriptor() == descriptor_before
fingerprint_unchanged = model.model_state_semantic_fingerprint == fingerprint_before
coefficients_unchanged = tuple(model.coefficients) == coefficients_before

print({
    "descriptor_unchanged": descriptor_unchanged,
    "fingerprint_unchanged": fingerprint_unchanged,
    "coefficients_unchanged": coefficients_unchanged,
})

{'descriptor_unchanged': True, 'fingerprint_unchanged': True, 'coefficients_unchanged': True}


## 12. Holdout and Model-Selection Isolation Audit
This notebook does not instrument live call counters, so the claims below are not runtime
counts taken from this kernel -- they are the audited result of the accompanying test suite,
`tests/test_forecasting_inference.py`, which:
- monkeypatches `pandas.read_csv` to fail the test immediately if any path ending in the final
  holdout filename is ever read during consumer loading or forecasting,
- monkeypatches the OLS fit entry point and the finalization entry point to fail the test
  immediately if either is ever called during consumer loading or forecasting,
- statically inspects (via `ast`) `scripts/forecasting_inference.py` and this notebook's code
  cells for any direct import or call of the model-selection or finalization entry points.

Separately, the authenticated final-model handoff loader *does* transitively replay the
Notebook 03 model-selection authentication chain as part of verifying upstream lineage. That is
allowed security validation of an already-made decision, not a new model-selection decision made
by Notebook 05.

In [30]:
holdout_and_selection_isolation_audit = {
    "direct_holdout_reads": 0,
    "consumer_fit_calls": 0,
    "consumer_model_selection_calls": 0,
    "evidence": "verified by tests/test_forecasting_inference.py (monkeypatch guards + AST source/notebook inspection)",
    "authenticated_upstream_selection_replay": "performed transitively inside the hardened final-model handoff loader; security validation, not a new Notebook 05 decision",
}
print(holdout_and_selection_isolation_audit)

{'direct_holdout_reads': 0, 'consumer_fit_calls': 0, 'consumer_model_selection_calls': 0, 'evidence': 'verified by tests/test_forecasting_inference.py (monkeypatch guards + AST source/notebook inspection)', 'authenticated_upstream_selection_replay': 'performed transitively inside the hardened final-model handoff loader; security validation, not a new Notebook 05 decision'}


## 13. Output Contract
Every forecast produced by this consumer is a `pandas.DataFrame` with exactly the columns
`["period", "forecast"]`, exactly 12 rows, canonical `YYYY-MM` period strings for `origin + 1`
through `origin + 12`, and finite forecast values in the bundle's declared unit. There is no
`y_true`, no error column, no confidence interval, and no score.

In [31]:
output_contract_report = {
    "row_count": len(canonical_forecast),
    "columns": list(canonical_forecast.columns),
    "future_period_semantics": bundle["output_contract"]["periods"],
    "unit": bundle["target"]["unit"],
    "all_finite": bool(np.isfinite(canonical_forecast["forecast"].to_numpy(dtype=float)).all()),
}
print(output_contract_report)

{'row_count': 12, 'columns': ['period', 'forecast'], 'future_period_semantics': 'origin+1 through origin+12', 'unit': 'degrees Fahrenheit', 'all_finite': True}


## 14. Final Artifact Immutability
The five final artifacts are re-hashed here and compared against the Section 2 snapshot. This
notebook never writes to `artifacts/models/nottem/`; it only reads, hashes, and validates.

In [32]:
artifact_hashes_after = artifact_hash_snapshot()
artifacts_unchanged = artifact_hashes_before == artifact_hashes_after
print({"artifacts_unchanged": artifacts_unchanged, "artifact_hashes_after": artifact_hashes_after})

{'artifacts_unchanged': True, 'artifact_hashes_after': {'final-pipeline.joblib': '3060d7a351688d7e60622d9112dbe01350e97964a01934b102beff912ae36f41', 'final-model-manifest.json': 'c6859e1425bbbf675ba169243e855ef153b6709c454341cdc75c1de14015fefa', 'final-test-evidence.json': 'a46f34bab5c652f186f5b4055d7b1d018823a631ae8f9f2cf1e042f248c1ba22', 'inference-bundle.json': 'a74d8d67256550694f12131a295a018511ae0ef476c12b6bb2dc72cb013b6079', 'final-model-handoff.json': '3a280491f5fa9e7d51db8e01c09dd202a2beddd3980f5074592fb44ceaa70732'}}


## 15. Fresh-Process Consumer Validation
The cell below spawns an independent Python process (not this kernel) that imports
`scripts.forecasting_inference` from scratch, loads the authenticated consumer, and produces one
12-row forecast from a synthetic one-row history. It depends on no state from this notebook's
kernel, from Notebook 04, or from pytest.

In [33]:
fresh_process_script = (
    "import pandas as pd\n"
    "from scripts.forecasting_inference import load_authenticated_forecasting_consumer, forecast_from_history\n"
    "fresh_consumer = load_authenticated_forecasting_consumer(project_root={project_root!r}, handoff_path={handoff_path!r})\n"
    "fresh_history = pd.DataFrame({{'period': ['1938-12'], 'temperature': [50.0]}})\n"
    "fresh_result = forecast_from_history(fresh_consumer, fresh_history)\n"
    "print('FRESH_PROCESS_ROWS', len(fresh_result))\n"
).format(project_root=str(PROJECT.root), handoff_path=HANDOFF_PATH)

fresh_process = subprocess.run(
    [sys.executable, "-c", fresh_process_script],
    cwd=str(PROJECT.root),
    capture_output=True,
    text=True,
    timeout=120,
)
fresh_process_ok = fresh_process.returncode == 0 and "FRESH_PROCESS_ROWS 12" in fresh_process.stdout
print({
    "returncode": fresh_process.returncode,
    "stdout": fresh_process.stdout.strip(),
    "fresh_process_ok": fresh_process_ok,
})

{'returncode': 0, 'stdout': 'FRESH_PROCESS_ROWS 12', 'fresh_process_ok': True}


## 16. Standalone Study Completion Readiness
`READY` below means only that the standalone scientific study (Notebooks 01-05) is complete and
that this inference consumer is validated -- it does **not** mean Atlas integration is complete
or that this model is operationally production-ready. `operational_modeling_ready` remains
`false` by design.

In [34]:
study_completion_checks = {
    "final_model_handoff_authenticated": True,
    "inference_bundle_authenticated": True,
    "trusted_model_loaded_with_sha_verification": True,
    "canonical_forecast_12_rows": len(canonical_forecast) == 12,
    "later_origin_forecast_12_rows": len(later_forecast) == 12,
    "outputs_finite": output_contract_report["all_finite"],
    "deterministic_repeatability": deterministic,
    "history_value_invariance_demonstrated": history_value_invariant,
    "model_state_immutable": fingerprint_unchanged and coefficients_unchanged,
    "final_artifacts_unchanged": artifacts_unchanged,
    "fresh_process_consumer_validated": fresh_process_ok,
    "operational_modeling_ready_is_false": handoff["readiness"]["operational_modeling_ready"] is False,
}

STUDY_COMPLETION_GATE = "READY" if all(study_completion_checks.values()) else "BLOCKED"

print({"study_completion_checks": study_completion_checks, "STUDY_COMPLETION_GATE": STUDY_COMPLETION_GATE})

{'study_completion_checks': {'final_model_handoff_authenticated': True, 'inference_bundle_authenticated': True, 'trusted_model_loaded_with_sha_verification': True, 'canonical_forecast_12_rows': True, 'later_origin_forecast_12_rows': True, 'outputs_finite': True, 'deterministic_repeatability': True, 'history_value_invariance_demonstrated': True, 'model_state_immutable': True, 'final_artifacts_unchanged': True, 'fresh_process_consumer_validated': True, 'operational_modeling_ready_is_false': True}, 'STUDY_COMPLETION_GATE': 'READY'}
